<a href="https://colab.research.google.com/github/trapti-singh/Data-integration/blob/main/linkedin_autoaaply.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip install playwright python-dotenv requests
!apt-get update && apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libxcomposite1
!playwright install chromium

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatk-bridge2.0-0 is already the newest version (2.38.0-3).
libatk1.0-0 is already the newest

In [34]:
import os, time, json, random, logging, requests
from datetime import datetime
from dotenv import load_dotenv


In [35]:
load_dotenv()

LOG_FILE = "easy_apply_log.txt"

# ── Logging setup ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

In [36]:
CONFIG = {
    # Credentials
    "li_email":       os.getenv("LI_EMAIL",      "trapti.collegedunia@gmail.com"),
    "li_password":    os.getenv("LI_PASSWORD",   "@@Anuradha##123"),


    # Job search
    "keywords": [
        "AI Engineer",
        "LLM Engineer",
        "Automation Engineer",
        "Generative AI Engineer",
        "n8n Developer",
        "AI Automation Specialist",
    ],
    "location":          "India",
    "remote_only":       True,
    "max_apply_per_run": 20,   # Total max applications per run
    "max_per_keyword":   5,    # Max per keyword

    # Relevance threshold (0-100) — Claude se match score
    "min_match_score": 60,

    # Tumhari profile — forms fill karne ke liye
    "profile": {
        "name":             "Trapti Singh",
        "email":            "singh.trapti2112@gmail.com",
        "phone":            "8999505687",
        "city":             "Mumbai",
        "state":            "Maharashtra",
        "country":          "India",
        "linkedin":         "https://linkedin.com/in/trapti-s-singh",
        "years_exp":        "4",
        "notice_period":    "Immediate",
        "current_ctc":      "Open to discussion",
        "expected_ctc":     "Open to discussion",
        "gender":           "Female",
        "authorized":       True,   # Authorized to work in India
        "needs_sponsorship":False,
        "relocate":         True,
        "cover_letter": (
            "Hi,\n\n"
            "I'm Trapti Singh, an AI Engineer with 4+ years of experience building "
            "LLM-powered applications and workflow automation systems. I specialize in "
            "Python, n8n, and generative AI integrations with a strong track record of "
            "shipping AI products from 0 to production in startup environments.\n\n"
            "I would love to bring this expertise to your team.\n\n"
            "Best regards,\nTrapti Singh\n+91 8999505687"
        ),
    },
}

In [37]:
def wait(lo=2, hi=4):
    time.sleep(random.uniform(lo, hi))


# ── Log helpers ──
def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                # If the file is empty or contains invalid JSON, return a default empty log
                return {"applied": [], "skipped": [], "failed": []}
    return {"applied": [], "skipped": [], "failed": []}


def save_log(data):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def already_done(job_id, log_data):
    done = [j.get("id") for j in log_data["applied"] + log_data["failed"]]
    return job_id in done


def applied_today_count(log_data):
    today = datetime.now().strftime("%Y-%m-%d")
    return len([j for j in log_data["applied"] if j.get("date", "").startswith(today)])


# ══════════════════════════════════════
#  STEP 1 — LOGIN
# ══════════════════════════════════════
def login(page):
    log.info("LinkedIn login...")
    page.goto("https://www.linkedin.com/login", wait_until="domcontentloaded")
    page.wait_for_selector("#username", timeout=15000)
    page.fill("#username", EMAIL)
    wait(0.5, 1)
    page.fill("#password", PASSWORD)
    wait(0.5, 1)
    page.click('[type="submit"]')

    try:
        page.wait_for_url("**/feed**", timeout=30000)
        log.info("Login successful!")
    except PWTimeout:
        log.warning("CAPTCHA/verification — browser mein manually solve karo")
        input("    Solve karo phir Enter dabaao: ")
    wait(2, 3)

In [38]:
def search_jobs(page):
    log.info(f"Searching: '{KEYWORD}' | Mid-Senior | Easy Apply...")

    # Direct URL with all filters:
    # f_E=3%2C4  = Experience: Associate(3) + Mid-Senior(4)
    # f_LF=f_AL  = Easy Apply only
    # sortBy=DD  = Most recent first
    url = (
        "https://www.linkedin.com/jobs/search/?"
        f"keywords={KEYWORD.replace(' ', '%20')}"
        "&location=India"
        "&f_E=3%2C4"
        "&f_LF=f_AL"
        "&sortBy=DD"
    )
    page.goto(url, wait_until="domcontentloaded")
    wait(3, 4)
    log.info("✅ Search results ready!")

In [39]:
def get_job_cards(page):
    jobs = []
    try:
        page.wait_for_selector(".jobs-search-results__list-item", timeout=10000)
    except PWTimeout:
        log.warning("Job list nahi mili")
        return jobs

    for card in page.query_selector_all(".jobs-search-results__list-item"):
        try:
            title_el   = card.query_selector(".job-card-list__title")
            company_el = card.query_selector(".job-card-container__primary-description")
            link_el    = card.query_selector("a[href*='/jobs/view/']")

            title   = title_el.inner_text().strip()    if title_el   else "Unknown"
            company = company_el.inner_text().strip()   if company_el else "Unknown"
            href    = link_el.get_attribute("href")     if link_el    else None

            if href:
                url    = f"https://www.linkedin.com{href}" if href.startswith("/") else href
                job_id = url.split("/view/")[1].split("/")[0].split("?")[0]
                jobs.append({"id": job_id, "title": title, "company": company, "url": url})
        except Exception:
            pass

    log.info(f"   {len(jobs)} jobs found on this page")
    return jobs

In [40]:
def get_label(el, page):
    """Element ka label text nikalo."""
    try:
        el_id = el.get_attribute("id")
        if el_id:
            lbl = page.query_selector(f"label[for='{el_id}']")
            if lbl:
                return lbl.inner_text().lower().strip()
        # Parent se dhundo
        for sel in ["label", "legend"]:
            try:
                parent = el.evaluate_handle("e => e.closest('.jobs-easy-apply-form-element, fieldset, div')")
                lbl = parent.query_selector(sel)
                if lbl:
                    t = lbl.inner_text().lower().strip()
                    if t:
                        return t
            except Exception:
                pass
    except Exception:
        pass
    return ""


def fill_inputs(page):
    for inp in page.query_selector_all(
        "input[type='text'], input[type='number'], input[type='tel'], input[type='email']"
    ):
        try:
            if not inp.is_visible():
                continue
            if (inp.input_value() or "").strip():
                continue  # Already filled — touch mat karo

            lbl = get_label(inp, page)

            if   any(k in lbl for k in ["phone", "mobile", "contact"]):
                inp.fill(PROFILE["phone"])
            elif "first name" in lbl:
                inp.fill(PROFILE["first_name"])
            elif "last name" in lbl:
                inp.fill(PROFILE["last_name"])
            elif any(k in lbl for k in ["full name", "your name"]):
                inp.fill(PROFILE["full_name"])
            elif "email" in lbl:
                inp.fill(PROFILE["email"])
            elif any(k in lbl for k in ["city", "current location"]):
                inp.fill(PROFILE["city"])
            elif any(k in lbl for k in ["year", "experience", "exp"]):
                inp.fill(PROFILE["years_exp"])
            elif any(k in lbl for k in ["notice", "joining", "availability"]):
                inp.fill(PROFILE["notice_period"])
            elif any(k in lbl for k in ["current ctc", "current salary"]):
                inp.fill(PROFILE["current_ctc"])
            elif any(k in lbl for k in ["expected", "desired salary"]):
                inp.fill(PROFILE["expected_ctc"])
            elif any(k in lbl for k in ["linkedin", "profile url"]):
                inp.fill(PROFILE["linkedin_url"])

            wait(0.2, 0.5)
        except Exception:
            pass


In [41]:
def fill_textareas(page):
    for ta in page.query_selector_all("textarea"):
        try:
            if ta.is_visible() and not (ta.input_value() or "").strip():
                ta.fill(PROFILE["cover_letter"])
                wait(0.2, 0.4)
        except Exception:
            pass


def fill_radios(page):
    for fieldset in page.query_selector_all("fieldset"):
        try:
            legend = fieldset.query_selector("legend")
            if not legend:
                continue
            q = legend.inner_text().lower()

            if   any(k in q for k in ["authorized", "legally", "right to work", "eligible"]):
                target = "yes"
            elif any(k in q for k in ["sponsor", "visa", "permit"]):
                target = "no"
            elif any(k in q for k in ["relocat"]):
                target = "yes"
            elif any(k in q for k in ["remote", "work from home"]):
                target = "yes"
            elif any(k in q for k in ["immediate"]):
                target = "yes"
            elif any(k in q for k in ["student", "currently enrolled"]):
                target = "no"
            elif any(k in q for k in ["disability", "veteran"]):
                target = "prefer"
            else:
                continue

            for radio in fieldset.query_selector_all("input[type='radio']"):
                rid  = radio.get_attribute("id") or ""
                rlbl = page.query_selector(f"label[for='{rid}']")
                rt   = (rlbl.inner_text().lower().strip() if rlbl else "")

                if (target == "yes"    and rt in ["yes", "true", "i am", "i do"]) or \
                   (target == "no"     and rt in ["no", "false", "i am not"]) or \
                   (target == "prefer" and "prefer" in rt):
                    radio.click()
                    wait(0.2, 0.4)
                    break
        except Exception:
            pass


def fill_dropdowns(page):
    for sel in page.query_selector_all("select"):
        try:
            if not sel.is_visible():
                continue
            lbl  = get_label(sel, page)
            opts = sel.evaluate(
                "el => [...el.options].map(o => ({v: o.value, t: o.text.toLowerCase()}))"
            )
            chosen = None

            if   any(k in lbl for k in ["country", "nation"]):
                for o in opts:
                    if "india" in o["t"]: chosen = o["v"]; break
            elif any(k in lbl for k in ["notice", "availability"]):
                for o in opts:
                    if any(k in o["t"] for k in ["immediate", "0 day", "no notice"]):
                        chosen = o["v"]; break
            elif any(k in lbl for k in ["experience", "year"]):
                for o in opts:
                    if "4" in o["t"] or "3-5" in o["t"]:
                        chosen = o["v"]; break
            elif "state" in lbl:
                for o in opts:
                    if "maharashtra" in o["t"]: chosen = o["v"]; break
            elif "gender" in lbl:
                for o in opts:
                    if PROFILE["gender"].lower() in o["t"]: chosen = o["v"]; break

            if chosen:
                sel.select_option(value=chosen)
                wait(0.2, 0.4)
        except Exception:
            pass


def fill_current_page(page):
    """Ek step ke saare fields fill karo."""
    fill_inputs(page)
    fill_radios(page)
    fill_dropdowns(page)
    fill_textareas(page)


# ══════════════════════════════════════
#  STEP 5 — NAVIGATE EASY APPLY MODAL
# ══════════════════════════════════════
def do_easy_apply(page):
    for step in range(1, 20):
        wait(1.5, 2.5)

        # Modal band ho gayi = submitted
        if not page.query_selector("[data-test-modal]"):
            return "submitted"

        log.info(f"   📋 Step {step}")

        # Fill all fields on this step
        fill_current_page(page)
        wait(0.5, 1)

        # Find button (Submit > Review > Continue > Next)
        btn = None
        btn_text = ""
        for label in ["Submit application", "Submit", "Review", "Continue", "Next"]:
            try:
                b = page.get_by_role("button", name=label, exact=False)
                if b.count() > 0 and b.first.is_visible() and b.first.is_enabled():
                    btn = b.first
                    btn_text = label
                    break
            except Exception:
                pass

        # Fallback
        if not btn:
            for b in page.query_selector_all("button"):
                try:
                    t = b.inner_text().strip().lower()
                    if any(k in t for k in ["submit", "review", "continue", "next"]):
                        if b.is_visible() and b.is_enabled():
                            btn = b; btn_text = t; break
                except Exception:
                    pass

        if not btn:
            log.warning(f"   Step {step} pe koi button nahi mila")
            return "stuck"

        log.info(f"   ➡️  '{btn_text}'")
        btn.click()
        wait(1.5, 2.5)

        # Submit ke baad check
        if "submit" in btn_text.lower():
            wait(2, 3)
            if not page.query_selector("[data-test-modal]"):
                return "submitted"
            try:
                body = page.inner_text("body").lower()
                if "application was sent" in body or "successfully applied" in body:
                    return "submitted"
            except Exception:
                pass
            return "submitted"

    return "max_steps"


def close_modal_safely(page):
    for sel in ["button[aria-label='Dismiss']", "button[data-test-modal-close-btn]"]:
        try:
            b = page.query_selector(sel)
            if b and b.is_visible():
                b.click(); wait(0.5, 1); break
        except Exception:
            pass
    for sel in ["button[data-test-dialog-primary-btn]", "button[aria-label='Discard']"]:
        try:
            b = page.query_selector(sel)
            if b and b.is_visible():
                b.click(); wait(0.5, 1); break
        except Exception:
            pass


In [46]:
# Global variables derived from CONFIG
# (These were causing NameErrors or were implicitly expected)
EMAIL = CONFIG["li_email"]
PASSWORD = CONFIG["li_password"]
MAX_APPLY = CONFIG["max_apply_per_run"]
PROFILE = CONFIG["profile"]
KEYWORD = CONFIG["keywords"][0] # Using the first keyword from the config list for the current run
LOCATION = CONFIG["location"] # Using location from config

# Import for async operations
import asyncio
import nest_asyncio
nest_asyncio.apply()

async def process_job(page, job, log_data):
    if already_done(job["id"], log_data):
        return "duplicate"

    log.info(f"\n{'─'*55}")
    log.info(f" {job['title']} @ {job['company']}")

    await page.goto(job["url"], wait_until="domcontentloaded")
    wait(2, 3)

    # Easy Apply button dhundo
    easy_btn = None
    for sel in [
        "button.jobs-apply-button",
        "button[aria-label*='Easy Apply']",
        ".jobs-s-apply button",
    ]:
        try:
            el = await page.wait_for_selector(sel, timeout=4000)
            if el and await el.is_visible() and "easy apply" in (await el.inner_text()).lower():
                easy_btn = el; break
        except PWTimeout:
            pass

    if not easy_btn:
        for b in await page.query_selector_all("button"):
            try:
                if "Easy Apply" in ((await b.inner_text()) or "") and await b.is_visible():
                    easy_btn = b; break
            except Exception:
                pass

    if not easy_btn:
        log.info("     Easy Apply nahi mila — next job")
        log_data["skipped"].append({**job, "reason": "no_easy_apply"})
        save_log(log_data)
        return "skip"

    log.info("   Easy Apply!")
    await easy_btn.click()
    wait(2, 3)

    try:
        await page.wait_for_selector("[data-test-modal]", timeout=8000)
    except PWTimeout:
        log.warning("   Modal nahi aaya")
        log_data["failed"].append({**job, "reason": "modal_timeout"})
        save_log(log_data)
        return "failed"

    result = await do_easy_apply(page)
    entry  = {
        **job,
        "result": result,
        "date":   datetime.now().strftime("%Y-%m-%d %H:%M")
    }

    if result == "submitted":
        log.info(f"   APPLIED! — {job['title']} @ {job['company']}")
        log_data["applied"].append(entry)
    else:
        log.warning(f"   Failed ({result})")
        log_data["failed"].append(entry)
        await close_modal_safely(page)

    save_log(log_data)
    return result

# ══════════════════════════════════════
#  STEP 1 — LOGIN
# ══════════════════════════════════════
async def login(page):
    log.info("LinkedIn login...")
    await page.goto("https://www.linkedin.com/login", wait_until="domcontentloaded")
    await page.wait_for_selector("#username", timeout=15000)
    await page.fill("#username", EMAIL)
    wait(0.5, 1)
    await page.fill("#password", PASSWORD)
    wait(0.5, 1)
    await page.click('[type="submit"]')

    try:
        await page.wait_for_url("**/feed**", timeout=30000)
        log.info("Login successful!")
    except PWTimeout:
        log.warning("CAPTCHA/verification — browser mein manually solve karo")
        input("    Solve karo phir Enter dabaao: ")
    wait(2, 3)

async def search_jobs(page):
    log.info(f"Searching: '{KEYWORD}' | Mid-Senior | Easy Apply...")

    # Direct URL with all filters:
    # f_E=3%2C4  = Experience: Associate(3) + Mid-Senior(4)
    # f_LF=f_AL  = Easy Apply only
    # sortBy=DD  = Most recent first
    url = (
        "https://www.linkedin.com/jobs/search/"
        f"?keywords={KEYWORD.replace(' ', '%20')}"
        "&location=India"
        "&f_E=3%2C4"
        "&f_LF=f_AL"
        "&sortBy=DD"
    )
    await page.goto(url, wait_until="domcontentloaded")
    wait(3, 4)
    log.info("✅ Search results ready!")

async def get_job_cards(page):
    jobs = []
    try:
        await page.wait_for_selector(".jobs-search-results__list-item", timeout=10000)
    except PWTimeout:
        log.warning("Job list nahi mili")
        return jobs

    for card in await page.query_selector_all(".jobs-search-results__list-item"):
        try:
            title_el   = await card.query_selector(".job-card-list__title")
            company_el = await card.query_selector(".job-card-container__primary-description")
            link_el    = await card.query_selector("a[href*='/jobs/view/']")

            title   = (await title_el.inner_text()).strip()    if title_el   else "Unknown"
            company = (await company_el.inner_text()).strip()   if company_el else "Unknown"
            href    = await link_el.get_attribute("href")     if link_el    else None

            if href:
                url    = f"https://www.linkedin.com{href}" if href.startswith("/") else href
                job_id = url.split("/view/")[1].split("/")[0].split("?")[0]
                jobs.append({"id": job_id, "title": title, "company": company, "url": url})
        except Exception:
            pass

    log.info(f"   {len(jobs)} jobs found on this page")
    return jobs

async def get_label(el, page):
    """Element ka label text nikalo."""
    try:
        el_id = await el.get_attribute("id")
        if el_id:
            lbl = await page.query_selector(f"label[for='{el_id}']")
            if lbl:
                return (await lbl.inner_text()).lower().strip()
        # Parent se dhundo
        for sel in ["label", "legend"]:
            try:
                parent = await el.evaluate_handle("e => e.closest('.jobs-easy-apply-form-element, fieldset, div')")
                lbl = await parent.query_selector(sel)
                if lbl:
                    t = (await lbl.inner_text()).lower().strip()
                    if t:
                        return t
            except Exception:
                pass
    except Exception:
        pass
    return ""


async def fill_inputs(page):
    for inp in await page.query_selector_all(
        "input[type='text'], input[type='number'], input[type='tel'], input[type='email']"
    ):
        try:
            if not await inp.is_visible():
                continue
            if ((await inp.input_value()) or "").strip():
                continue  # Already filled — touch mat karo

            lbl = await get_label(inp, page)

            if   any(k in lbl for k in ["phone", "mobile", "contact"]):
                await inp.fill(PROFILE["phone"])
            elif "first name" in lbl:
                await inp.fill(PROFILE["first_name"])
            elif "last name" in lbl:
                await inp.fill(PROFILE["last_name"])
            elif any(k in lbl for k in ["full name", "your name"]):
                await inp.fill(PROFILE["full_name"])
            elif "email" in lbl:
                await inp.fill(PROFILE["email"])
            elif any(k in lbl for k in ["city", "current location"]):
                await inp.fill(PROFILE["city"])
            elif any(k in lbl for k in ["year", "experience", "exp"]):
                await inp.fill(PROFILE["years_exp"])
            elif any(k in lbl for k in ["notice", "joining", "availability"]):
                await inp.fill(PROFILE["notice_period"])
            elif any(k in lbl for k in ["current ctc", "current salary"]):
                await inp.fill(PROFILE["current_ctc"])
            elif any(k in lbl for k in ["expected", "desired salary"]):
                await inp.fill(PROFILE["expected_ctc"])
            elif any(k in lbl for k in ["linkedin", "profile url"]):
                await inp.fill(PROFILE["linkedin_url"])

            wait(0.2, 0.5)
        except Exception:
            pass

async def fill_textareas(page):
    for ta in await page.query_selector_all("textarea"):
        try:
            if await ta.is_visible() and not ((await ta.input_value()) or "").strip():
                await ta.fill(PROFILE["cover_letter"])
                wait(0.2, 0.4)
        except Exception:
            pass


async def fill_radios(page):
    for fieldset in await page.query_selector_all("fieldset"):
        try:
            legend = await fieldset.query_selector("legend")
            if not legend:
                continue
            q = (await legend.inner_text()).lower()

            if   any(k in q for k in ["authorized", "legally", "right to work", "eligible"]):
                target = "yes"
            elif any(k in q for k in ["sponsor", "visa", "permit"]):
                target = "no"
            elif any(k in q for k in ["relocat"]):
                target = "yes"
            elif any(k in q for k in ["remote", "work from home"]):
                target = "yes"
            elif any(k in q for k in ["immediate"]):
                target = "yes"
            elif any(k in q for k in ["student", "currently enrolled"]):
                target = "no"
            elif any(k in q for k in ["disability", "veteran"]):
                target = "prefer"
            else:
                continue

            for radio in await fieldset.query_selector_all("input[type='radio']"):
                rid  = await radio.get_attribute("id") or ""
                rlbl = await page.query_selector(f"label[for='{rid}']")
                rt   = ((await rlbl.inner_text()).lower().strip() if rlbl else "")

                if (target == "yes"    and rt in ["yes", "true", "i am", "i do"]) or \
                   (target == "no"     and rt in ["no", "false", "i am not"]) or \
                   (target == "prefer" and "prefer" in rt):
                    await radio.click()
                    wait(0.2, 0.4)
                    break
        except Exception:
            pass


async def fill_dropdowns(page):
    for sel in await page.query_selector_all("select"):
        try:
            if not await sel.is_visible():
                continue
            lbl  = await get_label(sel, page)
            opts = await sel.evaluate(
                "el => [...el.options].map(o => ({v: o.value, t: o.text.toLowerCase()}))"
            )
            chosen = None

            if   any(k in lbl for k in ["country", "nation"]):
                for o in opts:
                    if "india" in o["t"]: chosen = o["v"]; break
            elif any(k in lbl for k in ["notice", "availability"]):
                for o in opts:
                    if any(k in o["t"] for k in ["immediate", "0 day", "no notice"]):
                        chosen = o["v"]; break
            elif any(k in lbl for k in ["experience", "year"]):
                for o in opts:
                    if "4" in o["t"] or "3-5" in o["t"]:
                        chosen = o["v"]; break
            elif "state" in lbl:
                for o in opts:
                    if "maharashtra" in o["t"]: chosen = o["v"]; break
            elif "gender" in lbl:
                for o in opts:
                    if PROFILE["gender"].lower() in o["t"]: chosen = o["v"]; break

            if chosen:
                await sel.select_option(value=chosen)
                wait(0.2, 0.4)
        except Exception:
            pass


async def fill_current_page(page):
    """Ek step ke saare fields fill karo."""
    await fill_inputs(page)
    await fill_radios(page)
    await fill_dropdowns(page)
    await fill_textareas(page)


# ══════════════════════════════════════
#  STEP 5 — NAVIGATE EASY APPLY MODAL
# ══════════════════════════════════════
async def do_easy_apply(page):
    for step in range(1, 20):
        wait(1.5, 2.5)

        # Modal band ho gayi = submitted
        if not await page.query_selector("[data-test-modal]"):
            return "submitted"

        log.info(f"   📋 Step {step}")

        # Fill all fields on this step
        await fill_current_page(page)
        wait(0.5, 1)

        # Find button (Submit > Review > Continue > Next)
        btn = None
        btn_text = ""
        for label in ["Submit application", "Submit", "Review", "Continue", "Next"]:
            try:
                b = page.get_by_role("button", name=label, exact=False)
                if (await b.count()) > 0 and (await b.first.is_visible()) and (await b.first.is_enabled()):
                    btn = b.first
                    btn_text = label
                    break
            except Exception:
                pass

        # Fallback
        if not btn:
            for b in await page.query_selector_all("button"):
                try:
                    t = (await b.inner_text()).strip().lower()
                    if any(k in t for k in ["submit", "review", "continue", "next"]):
                        if (await b.is_visible()) and (await b.is_enabled()):
                            btn = b; btn_text = t; break
                except Exception:
                    pass

        if not btn:
            log.warning(f"   Step {step} pe koi button nahi mila")
            return "stuck"

        log.info(f"   ➡️  '{btn_text}'")
        await btn.click()
        wait(1.5, 2.5)

        # Submit ke baad check
        if "submit" in btn_text.lower():
            wait(2, 3)
            if not await page.query_selector("[data-test-modal]"):
                return "submitted"
            try:
                body = (await page.inner_text("body")).lower()
                if "application was sent" in body or "successfully applied" in body:
                    return "submitted"
            except Exception:
                pass
            return "submitted"

    return "max_steps"


async def close_modal_safely(page):
    for sel in ["button[aria-label='Dismiss']", "button[data-test-modal-close-btn]"]:
        try:
            b = await page.query_selector(sel)
            if b and await b.is_visible():
                await b.click(); wait(0.5, 1); break
        except Exception:
            pass
    for sel in ["button[data-test-dialog-primary-btn]", "button[aria-label='Discard']"]:
        try:
            b = await page.query_selector(sel)
            if b and await b.is_visible():
                await b.click(); wait(0.5, 1); break
        except Exception:
            pass


async def run():
    log_data = load_log()
    done_today = applied_today_count(log_data)

    log.info("=" * 55)
    log.info("  LinkedIn Easy Apply Bot — Trapti Singh")
    log.info(f"  Keyword    : {KEYWORD}")
    log.info(f"  Filter     : Mid-Senior + Easy Apply")
    log.info(f"  Daily limit: {MAX_APPLY}")
    log.info(f"  Aaj applied: {done_today}/{MAX_APPLY}")
    log.info("=" * 55)

    if done_today >= MAX_APPLY:
        log.info(f"Aaj ka {MAX_APPLY} applications ka limit pura ho chuka hai!")
        log.info("    Kal phir chalao.")
        return

    remaining = MAX_APPLY - done_today

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=False,
            slow_mo=80,
            args=["--start-maximized"]
        )
        ctx = await browser.new_context(
            viewport={"width": 1366, "height": 768},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )
        page = await ctx.new_page()
        applied_now = 0

        try:
            await login(page)
            await search_jobs(page)

            page_num = 1
            while applied_now < remaining:
                log.info(f"\n📄 Page {page_num}")
                jobs = await get_job_cards(page)

                if not jobs:
                    log.info("Koi aur job nahi mili")
                    break

                for job in jobs:
                    if applied_now >= remaining:
                        break

                    result = await process_job(page, job, log_data)

                    if result == "submitted":
                        applied_now += 1
                        log.info(f"   Aaj: {done_today + applied_now}/{MAX_APPLY}")

                    wait(3, 6)

                # Next page
                page_num += 1
                try:
                    nxt = await page.query_selector(f"button[aria-label='Page {page_num}']")
                    if not nxt:
                        log.info("Aur pages nahi hain — done!")
                        break
                    await nxt.click()
                    wait(3, 4)
                except Exception:
                    break

        except KeyboardInterrupt:
            log.info("\n Rok diya")
        except Exception as e:
            log.error(f" Error: {e}", exc_info=True)
        finally:
            await browser.close()

    log.info(f"\n{'='*55}")
    log.info(f" DONE!")
    log.info(f" Aaj apply kiya : {applied_now}")
    log.info(f" Total applied  : {len(log_data['applied'])}")
    log.info(f" Failed         : {len(log_data['failed'])}")
    log.info(f" Log            : {LOG_FILE}")
    log.info(f"{'='*55}")


if __name__ == "__main__":
    asyncio.run(run())


TargetClosedError: BrowserType.launch: Target page, context or browser has been closed
Browser logs:

╔════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Looks like you launched a headed browser without having a XServer running.                     ║
║ Set either 'headless: true' or use 'xvfb-run <your-playwright-app>' before running Playwright. ║
║                                                                                                ║
║ <3 Playwright Team                                                                             ║
╚════════════════════════════════════════════════════════════════════════════════════════════════╝
Call log:
  - <launching> /root/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phish

In [54]:
import os, time, random, json, logging
from datetime import datetime
from dotenv import load_dotenv
from playwright.async_api import async_playwright, TimeoutError as PWTimeout # Changed to async_api
import asyncio # Added for async execution
import nest_asyncio # Added for Colab compatibility
nest_asyncio.apply() # Apply nest_asyncio

load_dotenv()

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler("bot_log.txt", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ══════════════════════════════════════
#  CONFIG — sirf yahan changes karo
# ══════════════════════════════════════
EMAIL     = os.getenv("LI_EMAIL",    "trapti.collegedunia@gmail.com")
PASSWORD  = os.getenv("LI_PASSWORD", "@@Anuradha##123")
KEYWORD   = "software testing"   # Job search keyword
MAX_APPLY = 15                    # Per day max applications
LOG_FILE  = "applied_jobs.json"

# Easy Apply form answers
PROFILE = {
    "first_name":     "Trapti",
    "last_name":      "Singh",
    "full_name":      "Trapti Singh",
    "email":          "singh.trapti2112@gmail.com",
    "phone":          "8999505687",
    "city":           "Mumbai",
    "state":          "Maharashtra",
    "country":        "India",
    "linkedin_url":   "https://linkedin.com/in/trapti-s-singh",
    "years_exp":      "4",
    "notice_period":  "Immediate",
    "current_ctc":    "4",
    "expected_ctc":   "7",
    "gender":         "Female",
    "cover_letter": (
        "Hi,\n\n"
        "I am Trapti Singh, an AI Engineer and Automation Specialist with 4+ years of "
        "experience in software testing, AI-driven QA, and workflow automation.\n\n"
        "I would love to bring my expertise to your team.\n\n"
        "Best regards,\nTrapti Singh\n+91 8999505687"
    ),
}
# ══════════════════════════════════════


def wait(lo=2, hi=4):
    time.sleep(random.uniform(lo, hi))


# ── Log helpers ──
def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return {"applied": [], "skipped": [], "failed": []}
    return {"applied": [], "skipped": [], "failed": []}


def save_log(data):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def already_done(job_id, log_data):
    done = [j.get("id") for j in log_data["applied"] + log_data["failed"]]
    return job_id in done


def applied_today_count(log_data):
    today = datetime.now().strftime("%Y-%m-%d")
    return len([j for j in log_data["applied"] if j.get("date", "").startswith(today)])


# ══════════════════════════════════════
#  STEP 1 — LOGIN
# ══════════════════════════════════════
async def login(page):
    log.info("🔐 LinkedIn login...")
    await page.goto("https://www.linkedin.com/login", wait_until="domcontentloaded")
    # Removed await page.wait_for_load_state('networkidle', timeout=30000)
    await page.wait_for_selector("#username", timeout=15000)
    await page.fill("#username", EMAIL)
    wait(0.5, 1)
    await page.fill("#password", PASSWORD)
    wait(0.5, 1)
    await page.click('[type="submit"]')

    try:
        await page.wait_for_url("**/feed**", timeout=30000)
        log.info("✅ Login successful!")
    except PWTimeout:
        log.warning("⚠️  CAPTCHA/verification — browser mein manually solve karo")
        # Removed input() as manual solving is not possible in headless mode
        raise # Re-raise the timeout to signal login failure
    wait(2, 3)


# ══════════════════════════════════════
#  STEP 2 — SEARCH + FILTERS
# ══════════════════════════════════════
async def search_jobs(page):
    log.info(f"🔎 Searching: '{KEYWORD}' | Mid-Senior | Easy Apply...")

    # Direct URL with all filters:
    # f_E=3%2C4  = Experience: Associate(3) + Mid-Senior(4)
    # f_LF=f_AL  = Easy Apply only
    # sortBy=DD  = Most recent first
    url = (
        "https://www.linkedin.com/jobs/search/"
        f"?keywords={KEYWORD.replace(' ', '%20')}"
        "&location=India"
        "&f_E=3%2C4"
        "&f_LF=f_AL"
        "&sortBy=DD"
    )
    await page.goto(url, wait_until="domcontentloaded")
    wait(3, 4)
    log.info("✅ Search results ready!")


# ══════════════════════════════════════
#  STEP 3 — GET JOB CARDS FROM PAGE
# ══════════════════════════════════════
async def get_job_cards(page):
    jobs = []
    try:
        await page.wait_for_selector(".jobs-search-results__list-item", timeout=10000)
    except PWTimeout:
        log.warning("Job list nahi mili")
        return jobs

    for card in await page.query_selector_all(".jobs-search-results__list-item"):
        try:
            title_el   = await card.query_selector(".job-card-list__title")
            company_el = await card.query_selector(".job-card-container__primary-description")
            link_el    = await card.query_selector("a[href*='/jobs/view/']")

            title   = (await title_el.inner_text()).strip()    if title_el   else "Unknown"
            company = (await company_el.inner_text()).strip()   if company_el else "Unknown"
            href    = await link_el.get_attribute("href")     if link_el    else None

            if href:
                url    = f"https://www.linkedin.com{href}" if href.startswith("/") else href
                job_id = url.split("/view/")[1].split("/")[0].split("?")[0]
                jobs.append({"id": job_id, "title": title, "company": company, "url": url})
        except Exception:
            pass

    log.info(f"   {len(jobs)} jobs found on this page")
    return jobs


# ══════════════════════════════════════
#  STEP 4 — FILL FORM FIELDS
# ══════════════════════════════════════
async def get_label(el, page):
    """Element ka label text nikalo."""
    try:
        el_id = await el.get_attribute("id")
        if el_id:
            lbl = await page.query_selector(f"label[for='{el_id}']")
            if lbl:
                return (await lbl.inner_text()).lower().strip()
        # Parent se dhundo
        for sel in ["label", "legend"]:
            try:
                parent = await el.evaluate_handle("e => e.closest('.jobs-easy-apply-form-element, fieldset, div')")
                lbl = await parent.query_selector(sel)
                if lbl:
                    t = (await lbl.inner_text()).lower().strip()
                    if t:
                        return t
            except Exception:
                pass
    except Exception:
        pass
    return ""


async def fill_inputs(page):
    for inp in await page.query_selector_all(
        "input[type='text'], input[type='number'], input[type='tel'], input[type='email']"
    ):
        try:
            if not await inp.is_visible():
                continue
            if ((await inp.input_value()) or "").strip():
                continue  # Already filled — touch mat karo

            lbl = await get_label(inp, page)

            if   any(k in lbl for k in ["phone", "mobile", "contact"]):
                await inp.fill(PROFILE["phone"])
            elif "first name" in lbl:
                await inp.fill(PROFILE["first_name"])
            elif "last name" in lbl:
                await inp.fill(PROFILE["last_name"])
            elif any(k in lbl for k in ["full name", "your name"]):
                await inp.fill(PROFILE["full_name"])
            elif "email" in lbl:
                await inp.fill(PROFILE["email"])
            elif any(k in lbl for k in ["city", "current location"]):
                await inp.fill(PROFILE["city"])
            elif any(k in lbl for k in ["year", "experience", "exp"]):
                await inp.fill(PROFILE["years_exp"])
            elif any(k in lbl for k in ["notice", "joining", "availability"]):
                await inp.fill(PROFILE["notice_period"])
            elif any(k in lbl for k in ["current ctc", "current salary"]):
                await inp.fill(PROFILE["current_ctc"])
            elif any(k in lbl for k in ["expected", "desired salary"]):
                await inp.fill(PROFILE["expected_ctc"])
            elif any(k in lbl for k in ["linkedin", "profile url"]):
                await inp.fill(PROFILE["linkedin_url"])

            wait(0.2, 0.5)
        except Exception:
            pass


async def fill_textareas(page):
    for ta in await page.query_selector_all("textarea"):
        try:
            if await ta.is_visible() and not ((await ta.input_value()) or "").strip():
                await ta.fill(PROFILE["cover_letter"])
                wait(0.2, 0.4)
        except Exception:
            pass


async def fill_radios(page):
    for fieldset in await page.query_selector_all("fieldset"):
        try:
            legend = await fieldset.query_selector("legend")
            if not legend:
                continue
            q = (await legend.inner_text()).lower()

            if   any(k in q for k in ["authorized", "legally", "right to work", "eligible"]):
                target = "yes"
            elif any(k in q for k in ["sponsor", "visa", "permit"]):
                target = "no"
            elif any(k in q for k in ["relocat"]):
                target = "yes"
            elif any(k in q for k in ["remote", "work from home"]):
                target = "yes"
            elif any(k in q for k in ["immediate"]):
                target = "yes"
            elif any(k in q for k in ["student", "currently enrolled"]):
                target = "no"
            elif any(k in q for k in ["disability", "veteran"]):
                target = "prefer"
            else:
                continue

            for radio in await fieldset.query_selector_all("input[type='radio']"):
                rid  = await radio.get_attribute("id") or ""
                rlbl = await page.query_selector(f"label[for='{rid}']")
                rt   = ((await rlbl.inner_text()).lower().strip() if rlbl else "")

                if (target == "yes"    and rt in ["yes", "true", "i am", "i do"]) or \
                   (target == "no"     and rt in ["no", "false", "i am not"]) or \
                   (target == "prefer" and "prefer" in rt):
                    await radio.click()
                    wait(0.2, 0.4)
                    break
        except Exception:
            pass


async def fill_dropdowns(page):
    for sel in await page.query_selector_all("select"):
        try:
            if not await sel.is_visible():
                continue
            lbl  = await get_label(sel, page)
            opts = await sel.evaluate(
                "el => [...el.options].map(o => ({v: o.value, t: o.text.toLowerCase()}))"
            )
            chosen = None

            if   any(k in lbl for k in ["country", "nation"]):
                for o in opts:
                    if "india" in o["t"] : chosen = o["v"]; break
            elif any(k in lbl for k in ["notice", "availability"]):
                for o in opts:
                    if any(k in o["t"] for k in ["immediate", "0 day", "no notice"]):
                        chosen = o["v"]; break
            elif any(k in lbl for k in ["experience", "year"]):
                for o in opts:
                    if "4" in o["t"] or "3-5" in o["t"]:
                        chosen = o["v"]; break
            elif "state" in lbl:
                for o in opts:
                    if "maharashtra" in o["t"] : chosen = o["v"]; break
            elif "gender" in lbl:
                for o in opts:
                    if PROFILE["gender"].lower() in o["t"] : chosen = o["v"]; break

            if chosen:
                await sel.select_option(value=chosen)
                wait(0.2, 0.4)
        except Exception:
            pass


async def fill_current_page(page):
    """Ek step ke saare fields fill karo."""
    await fill_inputs(page)
    await fill_radios(page)
    await fill_dropdowns(page)
    await fill_textareas(page)


# ══════════════════════════════════════
#  STEP 5 — NAVIGATE EASY APPLY MODAL
# ══════════════════════════════════════
async def do_easy_apply(page):
    for step in range(1, 20):
        wait(1.5, 2.5)

        # Modal band ho gayi = submitted
        if not await page.query_selector("[data-test-modal]"):
            return "submitted"

        log.info(f"   📋 Step {step}")

        # Fill all fields on this step
        await fill_current_page(page)
        wait(0.5, 1)

        # Find button (Submit > Review > Continue > Next)
        btn = None
        btn_text = ""
        for label in ["Submit application", "Submit", "Review", "Continue", "Next"]:
            try:
                b = page.get_by_role("button", name=label, exact=False)
                if (await b.count()) > 0 and (await b.first.is_visible()) and (await b.first.is_enabled()):
                    btn = b.first
                    btn_text = label
                    break
            except Exception:
                pass

        # Fallback
        if not btn:
            for b in await page.query_selector_all("button"):
                try:
                    t = (await b.inner_text()).strip().lower()
                    if any(k in t for k in ["submit", "review", "continue", "next"]):
                        if (await b.is_visible()) and (await b.is_enabled()):
                            btn = b; btn_text = t; break
                except Exception:
                    pass

        if not btn:
            log.warning(f"   Step {step} pe koi button nahi mila")
            return "stuck"

        log.info(f"   ➡️  '{btn_text}'")
        await btn.click()
        wait(1.5, 2.5)

        # Submit ke baad check
        if "submit" in btn_text.lower():
            wait(2, 3)
            if not await page.query_selector("[data-test-modal]"):
                return "submitted"
            try:
                body = (await page.inner_text("body")).lower()
                if "application was sent" in body or "successfully applied" in body:
                    return "submitted"
            except Exception:
                pass
            return "submitted"

    return "max_steps"


async def close_modal_safely(page):
    for sel in ["button[aria-label='Dismiss']", "button[data-test-modal-close-btn]"]:
        try:
            b = await page.query_selector(sel)
            if b and await b.is_visible():
                await b.click(); wait(0.5, 1); break
        except Exception:
            pass
    for sel in ["button[data-test-dialog-primary-btn]", "button[aria-label='Discard']"]:
        try:
            b = await page.query_selector(sel)
            if b and await b.is_visible():
                await b.click(); wait(0.5, 1); break
        except Exception:
            pass


# ══════════════════════════════════════
#  PROCESS ONE JOB
# ══════════════════════════════════════
async def process_job(page, job, log_data):
    if already_done(job["id"], log_data):
        return "duplicate"

    log.info(f"\n{'─'*55}")
    log.info(f"💼 {job['title']} @ {job['company']}")

    await page.goto(job["url"], wait_until="domcontentloaded")
    wait(2, 3)

    # Easy Apply button dhundo
    easy_btn = None
    for sel in [
        "button.jobs-apply-button",
        "button[aria-label*='Easy Apply']",
        ".jobs-s-apply button",
    ]:
        try:
            el = await page.wait_for_selector(sel, timeout=4000)
            if el and await el.is_visible() and "easy apply" in (await el.inner_text()).lower():
                easy_btn = el; break
        except PWTimeout:
            pass

    if not easy_btn:
        for b in await page.query_selector_all("button"):
            try:
                if "Easy Apply" in ((await b.inner_text()) or "") and await b.is_visible():
                    easy_btn = b; break
            except Exception:
                pass

    if not easy_btn:
        log.info("   ⚠️  Easy Apply nahi mila — next job")
        log_data["skipped"].append({**job, "reason": "no_easy_apply"})
        save_log(log_data)
        return "skip"

    log.info("   ⚡ Easy Apply!")
    await easy_btn.click()
    wait(2, 3)

    try:
        await page.wait_for_selector("[data-test-modal]", timeout=8000)
    except PWTimeout:
        log.warning("   Modal nahi aaya")
        log_data["failed"].append({**job, "reason": "modal_timeout"})
        save_log(log_data)
        return "failed"

    result = await do_easy_apply(page)
    entry  = {
        **job,
        "result": result,
        "date":   datetime.now().strftime("%Y-%m-%d %H:%M")
    }

    if result == "submitted":
        log.info(f"   🎉 APPLIED! — {job['title']} @ {job['company']}")
        log_data["applied"].append(entry)
    else:
        log.warning(f"   ❌ Failed ({result})")
        log_data["failed"].append(entry)
        await close_modal_safely(page)

    save_log(log_data)
    return result


# ══════════════════════════════════════
#  MAIN
# ══════════════════════════════════════
async def run():
    log_data = load_log()
    done_today = applied_today_count(log_data)

    log.info("=" * 55)
    log.info("  🤖 LinkedIn Easy Apply Bot — Trapti Singh")
    log.info(f"  Keyword    : {KEYWORD}")
    log.info(f"  Filter     : Mid-Senior + Easy Apply")
    log.info(f"  Daily limit: {MAX_APPLY}")
    log.info(f"  Aaj applied: {done_today}/{MAX_APPLY}")
    log.info("=" * 55)

    if done_today >= MAX_APPLY:
        log.info(f"🏁 Aaj ka {MAX_APPLY} applications ka limit pura ho chuka hai!")
        log.info("    Kal phir chalao.")
        return

    remaining = MAX_APPLY - done_today

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True, # Changed to True
            slow_mo=80,
            args=["--start-maximized"]
        )
        ctx = await browser.new_context(
            viewport={"width": 1366, "height": 768},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )
        page = await ctx.new_page()
        applied_now = 0

        try:
            await login(page)
            await search_jobs(page)

            page_num = 1
            while applied_now < remaining:
                log.info(f"\n📄 Page {page_num}")
                jobs = await get_job_cards(page)

                if not jobs:
                    log.info("Koi aur job nahi mili")
                    break

                for job in jobs:
                    if applied_now >= remaining:
                        break

                    result = await process_job(page, job, log_data)

                    if result == "submitted":
                        applied_now += 1
                        log.info(f"   📊 Aaj: {done_today + applied_now}/{MAX_APPLY}")

                    wait(3, 6)

                # Next page
                page_num += 1
                try:
                    nxt = await page.query_selector(f"button[aria-label='Page {page_num}']")
                    if not nxt:
                        log.info("Aur pages nahi hain — done!")
                        break
                    await nxt.click()
                    wait(3, 4)
                except Exception:
                    break

        except KeyboardInterrupt:
            log.info("\n⛔ Rok diya")
        except Exception as e:
            log.error(f"❌ Error: {e}", exc_info=True)
        finally:
            await browser.close()

    log.info(f"\n{'='*55}")
    log.info(f"🏁 DONE!")
    log.info(f"   ✅ Aaj apply kiya : {applied_now}")
    log.info(f"   📊 Total applied  : {len(log_data['applied'])}")
    log.info(f"   ❌ Failed         : {len(log_data['failed'])}")
    log.info(f"   📋 Log            : {LOG_FILE}")
    log.info(f"{'='*55}")


if __name__ == "__main__":
    asyncio.run(run())

ERROR:__main__:❌ Error: Timeout 30000ms exceeded.
=========================== logs ===========================
waiting for navigation to "**/feed**" until 'load'
Traceback (most recent call last):
  File "/tmp/ipykernel_727/1465269030.py", line 523, in run
    await login(page)
  File "/tmp/ipykernel_727/1465269030.py", line 104, in login
    await page.wait_for_url("**/feed**", timeout=30000)
  File "/usr/local/lib/python3.12/dist-packages/playwright/async_api/_generated.py", line 9186, in wait_for_url
    await self._impl_obj.wait_for_url(
  File "/usr/local/lib/python3.12/dist-packages/playwright/_impl/_page.py", line 580, in wait_for_url
    return await self._main_frame.wait_for_url(**locals_to_params(locals()))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/playwright/_impl/_frame.py", line 263, in wait_for_url
    async with self.expect_navigation(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lo